# E-commerce Marketplace Financial Analysis

Analyzing sales data and payment status from marketplace

In [55]:
import pandas as pd
import numpy as np

# Load the CSV files with semicolon delimiter and handle German number format
df_gmu = pd.read_csv(
    'report_booking_gmu_de_9de5e3ee9d9d950a2afba4b21aa6029fc8616843ea00a471bc76f03ed5d8c2e8.csv',
    sep=';',
    decimal=',',
    thousands='.',
    encoding='utf-8'
)

df_10000 = pd.read_csv(
    'report_booking10000_de_df2ac594159928a30cbdd344e7772459a8d196693f3c96288873dc316ef2bab5.csv',
    sep=';',
    decimal=',',
    thousands='.',
    encoding='utf-8'
)

# Convert dates to datetime
df_gmu['booking_date'] = pd.to_datetime(df_gmu['booking_date'])
df_10000['Datum'] = pd.to_datetime(df_10000['Datum'])

# Add month column
df_gmu['month'] = df_gmu['booking_date'].dt.to_period('M')
df_10000['month'] = df_10000['Datum'].dt.to_period('M')

print("GMU Dataset loaded")
print(f"Shape: {df_gmu.shape}")
print(f"Date range: {df_gmu['booking_date'].min()} to {df_gmu['booking_date'].max()}")

print("\n" + "="*50 + "\n")

print("10000 Dataset loaded")
print(f"Shape: {df_10000.shape}")
print(f"Date range: {df_10000['Datum'].min()} to {df_10000['Datum'].max()}")

GMU Dataset loaded
Shape: (44, 54)
Date range: 2025-08-05 08:53:44 to 2025-10-27 08:37:17


10000 Dataset loaded
Shape: (101, 9)
Date range: 2025-08-05 00:00:00 to 2025-10-25 00:00:00


In [56]:
# Understanding the data structure
print("="*70)
print("UNDERSTANDING THE DATA STRUCTURE")
print("="*70)

# GMU dataset analysis
print("\n1. GMU Dataset - Transaction Types:")
print(df_gmu['booking_text'].value_counts())

print("\n2. GMU Dataset - Orders with NaN order_number:")
nan_orders = df_gmu[df_gmu['order_number'].isna()]
print(f"Total rows with NaN order_number: {len(nan_orders)}")
print("\nBooking types with NaN order_number:")
print(nan_orders['booking_text'].value_counts())

print("\n3. GMU Dataset - Orders WITH order_number (actual sales):")
orders_with_number = df_gmu[df_gmu['order_number'].notna()]
print(f"Total rows with order_number: {len(orders_with_number)}")
print("\nBooking types with order_number:")
print(orders_with_number['booking_text'].value_counts())

UNDERSTANDING THE DATA STRUCTURE

1. GMU Dataset - Transaction Types:
booking_text
Payout                                                                                                                10
Fees for cancelled orders Juli 25                                                                                      2
Bezahlung Grundgebühr                                                                                                  2
Freigabe Verkaufserlös zu Bestell-Nr. MG6CC7Q/314567987448311 Injusa 6v 606 Ce E230 S6/0.6 C1 Batterie Schwarz ...     1
Freigabe Verkaufserlös zu Bestell-Nr. MX6XA6Q/314567986969924 Safta Mufasa Big Triple Federmäppchen Grün Mann G...     1
Freigabe Verkaufserlös zu Bestell-Nr. M5QHW6Q/314567987029312 Harry Potter Regenschirm ? 97 cm Leichter Kuppel-...     1
Freigabe Verkaufserlös zu Bestell-Nr. MZPEE6Q/314567986723309 Federmäppchen mit Zubehör Peppa Pig Pretty flower...     1
Freigabe Verkaufserlös zu Bestell-Nr. MY7ZB7Q/314567987437097 Kinder-K

In [57]:
# Compare the key columns between datasets
print("="*70)
print("COMPARING KEYS: order_number (GMU) vs Bestellnummer (10000)")
print("="*70)

# Get unique order numbers from both datasets
gmu_orders = set(df_gmu['order_number'].dropna().unique())
df10000_orders = set(df_10000['Bestellnummer'].dropna().unique())

print(f"\nUnique orders in GMU (with order_number): {len(gmu_orders)}")
print(f"Unique orders in 10000 (Bestellnummer): {len(df10000_orders)}")

# Find matches
matched_orders = gmu_orders.intersection(df10000_orders)
print(f"\nMatched orders (in both datasets): {len(matched_orders)}")

# Find orders only in GMU (not yet paid)
only_in_gmu = gmu_orders - df10000_orders
print(f"Orders only in GMU (potentially unpaid): {len(only_in_gmu)}")

# Find orders only in 10000 (paid but not in GMU detail - unusual)
only_in_10000 = df10000_orders - gmu_orders
print(f"Orders only in 10000 (unusual): {len(only_in_10000)}")

# Show sample of matched orders
print("\n\nSample matched order numbers:")
print(list(matched_orders)[:10])

COMPARING KEYS: order_number (GMU) vs Bestellnummer (10000)

Unique orders in GMU (with order_number): 29
Unique orders in 10000 (Bestellnummer): 42

Matched orders (in both datasets): 29
Orders only in GMU (potentially unpaid): 0
Orders only in 10000 (unusual): 13


Sample matched order numbers:
['MSBUPUQ', 'MYG45FQ', 'MA7S6UQ', 'MGA7HUQ', 'M7HKFUQ', 'MZPEE6Q', 'MDG6C6Q', 'MD6X67Q', 'MDA4CUQ', 'MAC3F6Q']


In [58]:
# Create sales dataset with payment status
print("="*70)
print("PAYMENT STATUS ANALYSIS")
print("="*70)

# Get actual sales transactions from GMU (with order_number)
gmu_sales = df_gmu[df_gmu['order_number'].notna()].copy()

# Add payment status flag
gmu_sales['is_paid'] = gmu_sales['order_number'].isin(df10000_orders)

print(f"\nTotal sales transactions in GMU: {len(gmu_sales)}")
print(f"Paid transactions: {gmu_sales['is_paid'].sum()}")
print(f"Unpaid transactions: {(~gmu_sales['is_paid']).sum()}")

# Show breakdown by booking_text
print("\n\nPayment status by transaction type:")
payment_breakdown = pd.crosstab(
    gmu_sales['booking_text'], 
    gmu_sales['is_paid'], 
    margins=True
)
print(payment_breakdown)

# Sample of unpaid orders
if len(only_in_gmu) > 0:
    print("\n\nSample of UNPAID orders (in GMU but not in 10000):")
    unpaid_sample = gmu_sales[gmu_sales['order_number'].isin(list(only_in_gmu)[:5])]
    print(unpaid_sample[['booking_date', 'order_number', 'payout', 'booking_text', 'order_date']].to_string())

PAYMENT STATUS ANALYSIS

Total sales transactions in GMU: 29
Paid transactions: 29
Unpaid transactions: 0


Payment status by transaction type:
is_paid                                             True  All
booking_text                                                 
Freigabe Verkaufserlös zu Bestell-Nr. M13B7FQ/3...     1    1
Freigabe Verkaufserlös zu Bestell-Nr. M4G78TQ/3...     1    1
Freigabe Verkaufserlös zu Bestell-Nr. M5PC46Q/3...     1    1
Freigabe Verkaufserlös zu Bestell-Nr. M5QHW6Q/3...     1    1
Freigabe Verkaufserlös zu Bestell-Nr. M5WHG6Q/3...     1    1
Freigabe Verkaufserlös zu Bestell-Nr. M7HKFUQ/3...     1    1
Freigabe Verkaufserlös zu Bestell-Nr. MA7S6UQ/3...     1    1
Freigabe Verkaufserlös zu Bestell-Nr. MAC3F6Q/3...     1    1
Freigabe Verkaufserlös zu Bestell-Nr. MD6X67Q/3...     1    1
Freigabe Verkaufserlös zu Bestell-Nr. MDA4CUQ/3...     1    1
Freigabe Verkaufserlös zu Bestell-Nr. MDG6C6Q/3...     1    1
Freigabe Verkaufserlös zu Bestell-Nr. MG6CC7Q/3...

In [59]:
# DIAGNOSTIC: Check why unpaid orders might be empty
print("="*70)
print("DIAGNOSTIC - Investigating Unpaid Orders")
print("="*70)

print(f"\nTotal unique orders in GMU: {len(gmu_orders)}")
print(f"Total unique orders in df_10000: {len(df10000_orders)}")
print(f"Orders only in GMU (unpaid): {len(only_in_gmu)}")

# Check if the issue is that ALL orders are paid
if len(only_in_gmu) == 0:
    print("\n⚠️ FINDING: ALL orders in GMU appear in df_10000 (all orders are marked as paid)")
    print("This could mean:")
    print("  - All sales have been paid already")
    print("  - OR the matching logic needs adjustment")
    
    # Let's check recent orders in GMU
    recent_gmu = df_gmu[df_gmu['order_number'].notna()].sort_values('booking_date', ascending=False).head(10)
    print("\n\nMost recent orders in GMU:")
    print(recent_gmu[['booking_date', 'order_number', 'booking_text']].to_string())
    
    # Check if those appear in df_10000
    print("\n\nChecking if these recent orders are in df_10000:")
    for order in recent_gmu['order_number'].head(5):
        in_10000 = order in df10000_orders
        print(f"  {order}: {'YES - in df_10000' if in_10000 else 'NO - not in df_10000'}")
else:
    print(f"\n✓ Found {len(only_in_gmu)} unpaid orders")
    print("\nSample unpaid order numbers:")
    print(list(only_in_gmu)[:10])

DIAGNOSTIC - Investigating Unpaid Orders

Total unique orders in GMU: 29
Total unique orders in df_10000: 42
Orders only in GMU (unpaid): 0

⚠️ FINDING: ALL orders in GMU appear in df_10000 (all orders are marked as paid)
This could mean:
  - All sales have been paid already
  - OR the matching logic needs adjustment


Most recent orders in GMU:
          booking_date order_number                                                                                                        booking_text
42 2025-10-23 18:22:56      MYG45FQ                       Freigabe Verkaufserlös zu Bestell-Nr. MYG45FQ/314567988204786 ADIDAS TIRO DU M TASCHE schwarz
40 2025-10-19 22:22:32      MD6X67Q  Freigabe Verkaufserlös zu Bestell-Nr. MD6X67Q/314567987579806 CERDÁ LIFE'S LITTLE MOMENTS Mädchen Kleiner Kinde...
39 2025-10-18 09:54:27      MKSQY7Q                   Freigabe Verkaufserlös zu Bestell-Nr. MKSQY7Q/314567987913119 Nike Rucksäcke Air Jordan, 9B050323
37 2025-10-07 17:23:23      M13B7FQ  Freigab

In [60]:
# Comprehensive financial summary
print("="*70)
print("COMPREHENSIVE FINANCIAL SUMMARY")
print("="*70)

# Calculate paid vs unpaid amounts
paid_orders = gmu_sales[gmu_sales['is_paid']]
unpaid_orders = gmu_sales[~gmu_sales['is_paid']]

summary = {
    'Total Sales (GMU)': {
        'Gross Sales': gmu_sales['price_gross'].sum(),
        'Shipping': gmu_sales['shipping_charges_gross'].sum(),
        'Total Gross': gmu_sales['sum_price_gross'].sum(),
        'Commission (Fees)': gmu_sales['fee_gross'].sum(),
        'Net Payout Expected': gmu_sales['payout'].sum(),
        'Order Count': len(gmu_sales)
    },
    'PAID Orders': {
        'Gross Sales': paid_orders['price_gross'].sum(),
        'Shipping': paid_orders['shipping_charges_gross'].sum(),
        'Total Gross': paid_orders['sum_price_gross'].sum(),
        'Commission (Fees)': paid_orders['fee_gross'].sum(),
        'Net Payout Expected': paid_orders['payout'].sum(),
        'Order Count': len(paid_orders)
    },
    'UNPAID Orders (Pending)': {
        'Gross Sales': unpaid_orders['price_gross'].sum(),
        'Shipping': unpaid_orders['shipping_charges_gross'].sum(),
        'Total Gross': unpaid_orders['sum_price_gross'].sum(),
        'Commission (Fees)': unpaid_orders['fee_gross'].sum(),
        'Net Payout Expected': unpaid_orders['payout'].sum(),
        'Order Count': len(unpaid_orders)
    }
}

summary_df = pd.DataFrame(summary).T
print("\n", summary_df.round(2))

# Calculate actual amount received from df_10000
actual_received = df_10000['Betrag'].sum()
print(f"\n\nActual Amount RECEIVED (df_10000): {actual_received:.2f} EUR")
print(f"Expected Payout for PAID orders (GMU): {paid_orders['payout'].sum():.2f} EUR")
print(f"Difference: {(actual_received - paid_orders['payout'].sum()):.2f} EUR")

# Commission analysis
avg_commission_rate = (gmu_sales['fee_gross'].sum() / gmu_sales['sum_price_gross'].sum() * 100)
print(f"\n\nAverage Commission Rate: {avg_commission_rate:.2f}%")
print(f"Total Commissions Paid: {gmu_sales['fee_gross'].sum():.2f} EUR")

COMPREHENSIVE FINANCIAL SUMMARY

                          Gross Sales  Shipping  Total Gross  \
Total Sales (GMU)             791.03     155.0       946.03   
PAID Orders                   791.03     155.0       946.03   
UNPAID Orders (Pending)         0.00       0.0         0.00   

                         Commission (Fees)  Net Payout Expected  Order Count  
Total Sales (GMU)                   128.02               818.01         29.0  
PAID Orders                         128.02               818.01         29.0  
UNPAID Orders (Pending)               0.00                 0.00          0.0  


Actual Amount RECEIVED (df_10000): 71.09 EUR
Expected Payout for PAID orders (GMU): 818.01 EUR
Difference: -746.92 EUR


Average Commission Rate: 13.53%
Total Commissions Paid: 128.02 EUR


In [61]:
# Monthly financial analysis
print("="*70)
print("MONTHLY FINANCIAL ANALYSIS")
print("="*70)

# Analyze GMU transactions
print("\n1. GMU - All Transactions by Month:")
gmu_monthly = df_gmu.groupby('month').agg({
    'payout': 'sum',
    'fee_gross': 'sum',
    'price_gross': 'sum',
    'order_number': 'count'
}).round(2)
gmu_monthly.columns = ['Total_Payout', 'Total_Fees', 'Total_Gross_Sales', 'Transaction_Count']
print(gmu_monthly)

# Analyze only sales (with order_number)
print("\n\n2. GMU - Sales Transactions Only (with order_number):")
gmu_sales_monthly = gmu_sales.groupby('month').agg({
    'payout': 'sum',
    'fee_gross': 'sum', 
    'price_gross': 'sum',
    'order_number': 'count'
}).round(2)
gmu_sales_monthly.columns = ['Sales_Payout', 'Sales_Fees', 'Sales_Gross', 'Sales_Count']
print(gmu_sales_monthly)

# Analyze payments received (df_10000)
print("\n\n3. Payments Actually Received (df_10000) by Month:")
df10000_monthly = df_10000.groupby('month').agg({
    'Betrag': 'sum',
    'Bestellnummer': 'count'
}).round(2)
df10000_monthly.columns = ['Amount_Received', 'Payment_Transactions']
print(df10000_monthly)

MONTHLY FINANCIAL ANALYSIS

1. GMU - All Transactions by Month:
         Total_Payout  Total_Fees  Total_Gross_Sales  Transaction_Count
month                                                                  
2025-08        317.56       47.96             320.52                  9
2025-09        370.24       59.37             343.61                 16
2025-10        130.21       20.69             126.90                  4


2. GMU - Sales Transactions Only (with order_number):
         Sales_Payout  Sales_Fees  Sales_Gross  Sales_Count
month                                                      
2025-08        317.56       47.96       320.52            9
2025-09        370.24       59.37       343.61           16
2025-10        130.21       20.69       126.90            4


3. Payments Actually Received (df_10000) by Month:
         Amount_Received  Payment_Transactions
month                                         
2025-08           -25.10                    39
2025-09          -162.25  

In [62]:
# Detailed monthly breakdown by transaction category
print("="*70)
print("DETAILED MONTHLY BREAKDOWN BY CATEGORY")
print("="*70)

# Categorize GMU transactions
def categorize_gmu_transaction(row):
    text = str(row['booking_text']).lower()
    if pd.isna(row['order_number']):
        # Non-order transactions
        if 'fee' in text or 'storno fee' in text:
            return 'Fees - Cancelled Orders'
        elif 'payout' in text:
            return 'Payout Transfer'
        elif 'grundgebühr' in text or 'bezahlung grundgebühr' in text:
            return 'Grundgebühr (Base Fee)'
        else:
            return 'Other Non-Sales'
    else:
        # Order transactions
        if 'freigabe' in text:
            return 'Sales - Released'
        else:
            return 'Sales - Other'

df_gmu['transaction_category'] = df_gmu.apply(categorize_gmu_transaction, axis=1)

print("\n1. GMU Transaction Categories:")
print(df_gmu['transaction_category'].value_counts())

print("\n\n2. Monthly Breakdown by Category (GMU):")
gmu_category_monthly = df_gmu.pivot_table(
    values='payout',
    index='month',
    columns='transaction_category',
    aggfunc='sum',
    fill_value=0
).round(2)
print(gmu_category_monthly)

# Categorize df_10000 transactions
def categorize_payment_transaction(row):
    text = str(row['Buchungstext']).lower()
    if 'wareneingang' in text:
        return 'Sales Income'
    elif 'provision' in text or 'netto provision' in text:
        return 'Commission/Fees'
    elif 'freigabe' in text:
        return 'Sales Released'
    elif 'payout' in text:
        return 'Payout Transfer'
    elif 'grundgebühr' in text or 'bezahlung grundgebühr' in text:
        return 'Grundgebühr (Base Fee)'
    elif 'storno' in text:
        return 'Cancellation/Refund'
    else:
        return 'Other'

df_10000['transaction_category'] = df_10000.apply(categorize_payment_transaction, axis=1)

print("\n\n3. df_10000 Transaction Categories:")
print(df_10000['transaction_category'].value_counts())

print("\n\n4. Monthly Breakdown by Category (df_10000 - Payments Received):")
df10000_category_monthly = df_10000.pivot_table(
    values='Betrag',
    index='month',
    columns='transaction_category',
    aggfunc='sum',
    fill_value=0
).round(2)
print(df10000_category_monthly)

DETAILED MONTHLY BREAKDOWN BY CATEGORY

1. GMU Transaction Categories:
transaction_category
Sales - Released           29
Payout Transfer            10
Fees - Cancelled Orders     3
Grundgebühr (Base Fee)      2
Name: count, dtype: int64


2. Monthly Breakdown by Category (GMU):
transaction_category  Fees - Cancelled Orders  Grundgebühr (Base Fee)  \
month                                                                   
2025-08                                     0                       0   
2025-09                                     0                       0   
2025-10                                     0                       0   

transaction_category  Payout Transfer  Sales - Released  
month                                                    
2025-08                             0            317.56  
2025-09                             0            370.24  
2025-10                             0            130.21  


3. df_10000 Transaction Categories:
transaction_category
Sales

In [63]:
# Export all analysis to Excel with multiple tabs (IMPROVED VERSION)
print("="*70)
print("EXPORTING TO EXCEL")
print("="*70)

# Create Excel writer
excel_filename = 'marketplace_financial_analysis.xlsx'
writer = pd.ExcelWriter(excel_filename, engine='xlsxwriter')

# Tab 1: Summary Report
print("\n1. Creating Summary Report tab...")
summary_df_export = summary_df.copy()
summary_df_export.to_excel(writer, sheet_name='1_Summary', startrow=0)

# Add additional summary info
additional_info = pd.DataFrame({
    'Metric': [
        'Actual Amount Received (from df_10000)',
        'Expected Payout for Paid Orders',
        'Difference',
        'Average Commission Rate (%)',
        'Total Commissions Paid',
        'Orders with Missing/Zero Amounts',
        'Unpaid Orders Count'
    ],
    'Value': [
        actual_received,
        paid_orders['payout'].sum(),
        actual_received - paid_orders['payout'].sum(),
        avg_commission_rate,
        gmu_sales['fee_gross'].sum(),
        len(problematic_orders),
        len(unpaid_orders)
    ]
})
additional_info.to_excel(writer, sheet_name='1_Summary', startrow=len(summary_df_export)+3, index=False)

# Tab 2: UNPAID Orders (Pending Payment)
print("2. Creating Unpaid Orders tab...")
if len(unpaid_orders) > 0:
    unpaid_orders_export = unpaid_orders[[
        'booking_date', 'order_date', 'order_number', 
        'title_item', 'price_gross', 'shipping_charges_gross', 
        'sum_price_gross', 'fee_gross', 'fee_%', 'payout',
        'buyer.email', 'shipping.first_name', 'shipping.last_name',
        'shipping.city', 'shipping.country', 'booking_text'
    ]].copy()
    
    # Sort by booking date (most recent first)
    unpaid_orders_export = unpaid_orders_export.sort_values('booking_date', ascending=False)
    
    # Add days since order
    unpaid_orders_export['days_since_order'] = (pd.Timestamp.now() - unpaid_orders_export['booking_date']).dt.days
    
    unpaid_orders_export.to_excel(writer, sheet_name='2_Unpaid_Orders', index=False)
else:
    # Create empty dataframe with message
    pd.DataFrame({'Message': ['No unpaid orders found - all orders have been paid']}).to_excel(
        writer, sheet_name='2_Unpaid_Orders', index=False
    )

# Tab 3: PAID Orders
print("3. Creating Paid Orders tab...")
paid_orders_export = paid_orders[[
    'booking_date', 'order_date', 'order_number',
    'title_item', 'price_gross', 'shipping_charges_gross',
    'sum_price_gross', 'fee_gross', 'fee_%', 'payout',
    'buyer.email', 'shipping.first_name', 'shipping.last_name',
    'shipping.city', 'shipping.country', 'booking_text'
]].copy()
paid_orders_export = paid_orders_export.sort_values('booking_date', ascending=False)
paid_orders_export.to_excel(writer, sheet_name='3_Paid_Orders', index=False)

# Tab 4: Monthly Sales by Category (GMU)
print("4. Creating Monthly Sales by Category tab...")
gmu_category_monthly_export = gmu_category_monthly.copy()
gmu_category_monthly_export.index = gmu_category_monthly_export.index.astype(str)
gmu_category_monthly_export.to_excel(writer, sheet_name='4_Monthly_GMU_Categories')

# Tab 5: Monthly Payments by Category (df_10000)
print("5. Creating Monthly Payments by Category tab...")
df10000_category_monthly_export = df10000_category_monthly.copy()
df10000_category_monthly_export.index = df10000_category_monthly_export.index.astype(str)
df10000_category_monthly_export.to_excel(writer, sheet_name='5_Monthly_Payment_Categories')

# Tab 6: Orders with Missing/Zero Amounts (ERRORS)
print("6. Creating Orders with Missing Amounts tab...")
if len(problematic_orders) > 0:
    problematic_export = problematic_orders[[
        'booking_date', 'order_date', 'order_number', 'booking_text',
        'title_item', 'price_gross', 'shipping_charges_gross',
        'sum_price_gross', 'fee_gross', 'payout',
        'has_missing_price', 'has_missing_payout', 'has_missing_sum'
    ]].copy()
    problematic_export = problematic_export.sort_values('booking_date', ascending=False)
    problematic_export.to_excel(writer, sheet_name='6_Missing_Amounts_ERROR', index=False)
else:
    pd.DataFrame({'Message': ['No orders with missing/zero amounts found']}).to_excel(
        writer, sheet_name='6_Missing_Amounts_ERROR', index=False
    )

# Tab 7: Payment Timing Analysis
print("7. Creating Payment Timing Analysis tab...")
paid_details_export = paid_details[[
    'order_number', 'booking_date', 'order_date', 'payment_date',
    'payment_delay_days', 'price_gross', 'fee_gross', 'payout',
    'title_item', 'buyer.email'
]].copy()
paid_details_export = paid_details_export.sort_values('payment_date', ascending=False)
paid_details_export.to_excel(writer, sheet_name='7_Payment_Timing', index=False)

# Tab 8: All GMU Transactions with Categories
print("8. Creating All GMU Transactions tab...")
df_gmu_export = df_gmu.copy()
df_gmu_export['month'] = df_gmu_export['month'].astype(str)
df_gmu_export.to_excel(writer, sheet_name='8_All_GMU_Transactions', index=False)

# Tab 9: All Payment Transactions (df_10000) with Categories
print("9. Creating All Payment Transactions tab...")
df_10000_export = df_10000.copy()
df_10000_export['month'] = df_10000_export['month'].astype(str)
df_10000_export = df_10000_export.sort_values('Datum', ascending=False)
df_10000_export.to_excel(writer, sheet_name='9_All_Payments_Received', index=False)

# Close the writer and save the file
writer.close()

print(f"\n✓ Excel file created successfully: {excel_filename}")
print(f"\nTabs created:")
print("  1. Summary - Overall financial summary")
print("  2. Unpaid_Orders - Orders awaiting payment from marketplace")
print("  3. Paid_Orders - Orders already paid")
print("  4. Monthly_GMU_Categories - Monthly breakdown by transaction type (GMU)")
print("  5. Monthly_Payment_Categories - Monthly breakdown by transaction type (Payments)")
print("  6. Missing_Amounts_ERROR - Orders with missing/zero amounts (marketplace errors)")
print("  7. Payment_Timing - Payment delay analysis")
print("  8. All_GMU_Transactions - Complete GMU dataset with categories")
print("  9. All_Payments_Received - Complete payment transactions with categories")

EXPORTING TO EXCEL

1. Creating Summary Report tab...


NameError: name 'problematic_orders' is not defined

In [ ]:
# Flag orders with missing or zero amounts (potential marketplace errors)
print("="*70)
print("FLAGGING ORDERS WITH MISSING/ZERO AMOUNTS")
print("="*70)

# Check GMU for missing/zero amounts
gmu_sales_check = df_gmu[df_gmu['order_number'].notna()].copy()

# Flag orders with issues
gmu_sales_check['has_missing_price'] = (gmu_sales_check['price_gross'].isna()) | (gmu_sales_check['price_gross'] == 0)
gmu_sales_check['has_missing_payout'] = (gmu_sales_check['payout'].isna()) | (gmu_sales_check['payout'] == 0)
gmu_sales_check['has_missing_sum'] = (gmu_sales_check['sum_price_gross'].isna()) | (gmu_sales_check['sum_price_gross'] == 0)

problematic_orders = gmu_sales_check[
    gmu_sales_check['has_missing_price'] | 
    gmu_sales_check['has_missing_payout'] | 
    gmu_sales_check['has_missing_sum']
]

print(f"\nTotal orders with amount issues: {len(problematic_orders)}")
print(f"  - Missing/zero price_gross: {gmu_sales_check['has_missing_price'].sum()}")
print(f"  - Missing/zero payout: {gmu_sales_check['has_missing_payout'].sum()}")
print(f"  - Missing/zero sum_price_gross: {gmu_sales_check['has_missing_sum'].sum()}")

if len(problematic_orders) > 0:
    print("\n\nSample of problematic orders:")
    print(problematic_orders[['booking_date', 'order_number', 'booking_text', 'price_gross', 'sum_price_gross', 'payout']].head(10).to_string())
else:
    print("\n✓ No orders with missing/zero amounts found in GMU")

# Check df_10000 for missing/zero amounts
df10000_check = df_10000.copy()
df10000_check['has_missing_amount'] = (df10000_check['Betrag'].isna()) | (df10000_check['Betrag'] == 0)

problematic_payments = df10000_check[df10000_check['has_missing_amount']]

print(f"\n\nPayment transactions with missing/zero amounts: {len(problematic_payments)}")
if len(problematic_payments) > 0:
    print("\nSample:")
    print(problematic_payments[['Datum', 'Bestellnummer', 'Buchungstext', 'Betrag']].head(10).to_string())
else:
    print("✓ No payment transactions with missing/zero amounts found")

In [ ]:
# Export all analysis to Excel with multiple tabs
print("="*70)
print("EXPORTING TO EXCEL")
print("="*70)

# Create Excel writer
excel_filename = 'marketplace_financial_analysis.xlsx'
writer = pd.ExcelWriter(excel_filename, engine='xlsxwriter')

# Tab 1: Summary Report
print("\n1. Creating Summary Report tab...")
summary_df_export = summary_df.copy()
summary_df_export.to_excel(writer, sheet_name='1_Summary', startrow=0)

# Add additional summary info
additional_info = pd.DataFrame({
    'Metric': [
        'Actual Amount Received (from df_10000)',
        'Expected Payout for Paid Orders',
        'Difference',
        'Average Commission Rate (%)',
        'Total Commissions Paid'
    ],
    'Value': [
        actual_received,
        paid_orders['payout'].sum(),
        actual_received - paid_orders['payout'].sum(),
        avg_commission_rate,
        gmu_sales['fee_gross'].sum()
    ]
})
additional_info.to_excel(writer, sheet_name='1_Summary', startrow=len(summary_df_export)+3, index=False)

# Tab 2: UNPAID Orders (Pending Payment)
print("2. Creating Unpaid Orders tab...")
unpaid_orders_export = unpaid_orders[[
    'booking_date', 'order_date', 'order_number', 
    'title_item', 'price_gross', 'shipping_charges_gross', 
    'sum_price_gross', 'fee_gross', 'fee_%', 'payout',
    'buyer.email', 'shipping.first_name', 'shipping.last_name',
    'shipping.city', 'shipping.country'
]].copy()

# Sort by booking date (most recent first)
unpaid_orders_export = unpaid_orders_export.sort_values('booking_date', ascending=False)

# Add days since order
unpaid_orders_export['days_since_order'] = (pd.Timestamp.now() - unpaid_orders_export['booking_date']).dt.days

unpaid_orders_export.to_excel(writer, sheet_name='2_Unpaid_Orders', index=False)

# Tab 3: PAID Orders
print("3. Creating Paid Orders tab...")
paid_orders_export = paid_orders[[
    'booking_date', 'order_date', 'order_number',
    'title_item', 'price_gross', 'shipping_charges_gross',
    'sum_price_gross', 'fee_gross', 'fee_%', 'payout',
    'buyer.email', 'shipping.first_name', 'shipping.last_name',
    'shipping.city', 'shipping.country'
]].copy()
paid_orders_export = paid_orders_export.sort_values('booking_date', ascending=False)
paid_orders_export.to_excel(writer, sheet_name='3_Paid_Orders', index=False)

# Tab 4: Monthly Sales Analysis
print("4. Creating Monthly Sales Analysis tab...")
gmu_sales_monthly_export = gmu_sales_monthly.copy()
gmu_sales_monthly_export.index = gmu_sales_monthly_export.index.astype(str)
gmu_sales_monthly_export.to_excel(writer, sheet_name='4_Monthly_Sales')

# Tab 5: Monthly Payments Received
print("5. Creating Monthly Payments Received tab...")
df10000_monthly_export = df10000_monthly.copy()
df10000_monthly_export.index = df10000_monthly_export.index.astype(str)
df10000_monthly_export.to_excel(writer, sheet_name='5_Monthly_Payments')

# Tab 6: Payment Timing Analysis
print("6. Creating Payment Timing Analysis tab...")
paid_details_export = paid_details[[
    'order_number', 'booking_date', 'order_date', 'payment_date',
    'payment_delay_days', 'price_gross', 'fee_gross', 'payout',
    'title_item', 'buyer.email'
]].copy()
paid_details_export = paid_details_export.sort_values('payment_date', ascending=False)
paid_details_export.to_excel(writer, sheet_name='6_Payment_Timing', index=False)

# Tab 7: All GMU Transactions
print("7. Creating All GMU Transactions tab...")
df_gmu_export = df_gmu.copy()
df_gmu_export['month'] = df_gmu_export['month'].astype(str)
df_gmu_export.to_excel(writer, sheet_name='7_All_GMU_Transactions', index=False)

# Tab 8: All Payment Transactions (df_10000)
print("8. Creating All Payment Transactions tab...")
df_10000_export = df_10000.copy()
df_10000_export['month'] = df_10000_export['month'].astype(str)
df_10000_export = df_10000_export.sort_values('Datum', ascending=False)
df_10000_export.to_excel(writer, sheet_name='8_All_Payments_Received', index=False)

# Close the writer and save the file
writer.close()

print(f"\n✓ Excel file created successfully: {excel_filename}")
print(f"\nTabs created:")
print("  1. Summary - Overall financial summary")
print("  2. Unpaid_Orders - Orders awaiting payment from marketplace")
print("  3. Paid_Orders - Orders already paid")
print("  4. Monthly_Sales - Monthly sales breakdown")
print("  5. Monthly_Payments - Monthly payments received")
print("  6. Payment_Timing - Payment delay analysis")
print("  7. All_GMU_Transactions - Complete GMU dataset")
print("  8. All_Payments_Received - Complete payment transactions")